#  Connexion & Préparation des Données

### Connecter Streamlit à financecore_db via SQLAlchemy

In [21]:
###Connexion postgresql
from sqlalchemy import create_engine,text
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

 #Récupération des paramètres
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
engine=create_engine( f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db_name}")
print("Connexion ok")

Connexion ok


In [22]:

df_agences=pd.read_sql("select * from agences",engine)
df_agences.head()

,id_agence,nom_agence
0,1,Marseille-Vieux-Port
1,2,Bordeaux-Meriadeck
2,3,Lyon-Part-Dieu
3,4,Lille-Grand-Place
4,5,Toulouse-Capitole


In [23]:
df_segments=pd.read_sql("select * from segments ",engine)
df_segments.head()

,id_segment,nom_segment
0,1,Premium
1,2,Risque
2,3,Standard


In [24]:
df_transactions=pd.read_sql("select * from transactions",engine)
df_transactions.head()

,id_transactions,id_compte,id_produit,id_agence,id_temps,montant,devise,montant_eur,type_operation,statut
0,TXN000559,1,1,1,1,2050.42,EUR,2050.42,Credit,Complete
1,TXN001154,2,2,2,2,-123.66,GBP,-143.79,Debit,Rejete
2,TXN000764,3,3,3,3,-396.17,EUR,-396.17,Debit,Complete
3,TXN001598,4,2,2,4,225.20,EUR,225.20,Credit,Complete
4,TXN001873,5,5,2,5,935.32,EUR,935.32,Credit,Complete


In [25]:
df_clients=pd.read_sql("select * from clients",engine)
df_clients.head()

,id_client,score_credit,categorie_risque,taux_rejet,id_segment
0,CLI0060,645,Medium,0.056,1
1,CLI0057,435,High,0.046,2
2,CLI0015,648,Medium,0.040,3
3,CLI0045,704,Low,0.046,3
4,CLI0034,457,High,0.046,2


In [26]:
df_comptes=pd.read_sql("select * from comptes ",engine)
df_comptes.head()

,id_compte,id_client,solde
0,1,CLI0060,16415.10
1,2,CLI0057,42890.81
2,3,CLI0015,48489.38
3,4,CLI0045,43962.51
4,5,CLI0034,17312.83


In [27]:
df_produits=pd.read_sql("select * from produits",engine)
df_produits.head()

,id_produit,nom_produit,categorie
0,1,Compte Epargne,Depot especes
1,2,Credit Consommation,Retrait DAB
2,3,PEA,Prelevement
3,4,Credit Consommation,Paiement CB
4,5,Credit Immobilier,Interets


In [28]:
df_temps=pd.read_sql("select * from temps",engine)
df_temps.head()

,id_temps,date_transaction,annee,mois,trimestre,jour_semaine
0,1,2022-04-19,2022,4,2,1
1,2,2024-06-20,2024,6,2,3
2,3,2024-08-28,2024,8,3,2
3,4,2024-01-07,2024,1,1,6
4,5,2024-08-11,2024,8,3,6


In [29]:
df=pd.merge(df_clients,df_segments,on="id_segment")
df=pd.merge(df,df_comptes,on="id_client")
df=pd.merge(df,df_transactions,on="id_compte")
df=pd.merge(df,df_agences,on="id_agence")
df=pd.merge(df,df_produits,on="id_produit")
df=pd.merge(df,df_temps,on="id_temps")

### Calculer les métriques agrégées nécessaires à chaque page du dashboard

##### 📊 Volume

In [30]:
#Nombre total de transactions
nbr_total_transaction=df["id_transactions"].count()
print("Nombre total de transactions: ",nbr_total_transaction)
#Nombre de transactions par client
nbr_total_tran_client=df.groupby("id_client")["id_transactions"].count()
print("Nombre de transactions par client: \n",nbr_total_tran_client)


Nombre total de transactions:  2000
Nombre de transactions par client: 
 id_client
CLI0001    14
CLI0002    12
CLI0003    13
CLI0004    16
CLI0005     9
           ..
CLI0146    10
CLI0147    11
CLI0148    18
CLI0149     6
CLI0150    10
Name: id_transactions, Length: 150, dtype: int64


In [31]:
#Nombre de transactions par PRODUIT 
nbr_total_tran_produit=df.groupby("nom_produit")["id_transactions"].count()
nbr_total_tran_produit

nom_produit
Assurance Vie          255
Compte Courant         251
Compte Epargne         273
Credit Auto            213
Credit Consommation    247
Credit Immobilier      272
Livret A               232
PEA                    257
Name: id_transactions, dtype: int64

In [32]:
#Nombre de transactions par agence 
nbr_total_tran_agence=df.groupby("nom_agence")["id_transactions"].count()
nbr_total_tran_agence

nom_agence
Bordeaux-Meriadeck      414
Lille-Grand-Place       214
Lyon-Part-Dieu          299
Marseille-Vieux-Port    323
Nantes-Commerce         200
Nice-Massena            131
Paris-Centre            228
Toulouse-Capitole       191
Name: id_transactions, dtype: int64

#### 💸 Montants

In [33]:
#Montant total des transactions
montant_total_credit = df[df["type_operation"] == "Credit"]["montant"].sum()
print("Montant total des transactions pour credit: ",montant_total_credit)
montant_total_debit = df[df["type_operation"] == "Debit"]["montant"].sum()
print("Montant total des transactions pour debit: ",montant_total_debit)
#Montant moyen par transaction
montant_moyen_transaction=df.groupby("id_transactions")["montant"].mean()
print("montant moyen par transactions: \n",montant_moyen_transaction)
#Montant médian
montant_median=df["montant"].median()
print("montant médian : ",montant_median)

#Montant max / min
montant_max=df["montant"].max()
print("montant max: ",montant_max)
montant_min=df["montant"].min()
print("montant min: ",montant_min)

Montant total des transactions pour credit:  1892534.95
Montant total des transactions pour debit:  -2326696.3
montant moyen par transactions: 
 id_transactions
TXN000001    -312.96
TXN000002     442.55
TXN000003    2286.59
TXN000004    -299.15
TXN000005    1998.20
              ...   
TXN001996    2353.67
TXN001997   -2248.80
TXN001998   -2132.99
TXN001999     170.22
TXN002000    -394.89
Name: montant, Length: 2000, dtype: float64
montant médian :  -88.49000000000001
montant max:  150000.0
montant min:  -200000.0


#### 📅 Temps

In [34]:
#Transactions par  mois / année
transactions_mois=df.groupby("mois")["id_transactions"].count()
print("transactions par mois: \n",transactions_mois)
transactions_annee=df.groupby("annee")["id_transactions"].count()
print("transactions par annee: \n",transactions_annee)
#Taux de croissance des transactions (%)
taux_croissance=transactions_mois.pct_change()*100
print("taux croissance des transactions: \n",taux_croissance.head())

transactions par mois: 
 mois
1     179
2     156
3     155
4     149
5     173
6     135
7     158
8     193
9     167
10    228
11    146
12    161
Name: id_transactions, dtype: int64
transactions par annee: 
 annee
2022    651
2023    729
2024    620
Name: id_transactions, dtype: int64
taux croissance des transactions: 
 mois
1          NaN
2   -12.849162
3    -0.641026
4    -3.870968
5    16.107383
Name: id_transactions, dtype: float64


### TOP 10 CLIENTS

In [35]:
top_10_clients=nbr_total_tran_client.sort_values(ascending=False).head(10)
top_10_clients

id_client
CLI0122    22
CLI0100    21
CLI0129    21
CLI0063    20
CLI0017    20
CLI0093    20
CLI0096    20
CLI0127    20
CLI0040    19
CLI0065    19
Name: id_transactions, dtype: int64

### TOP 5 PRODUITS

In [36]:
top_5_produits=nbr_total_tran_produit.sort_values(ascending=False).head()
top_5_produits

nom_produit
Compte Epargne       273
Credit Immobilier    272
PEA                  257
Assurance Vie        255
Compte Courant       251
Name: id_transactions, dtype: int64

### TOP  AGENCES

In [37]:
top_agences=nbr_total_tran_agence.sort_values(ascending=False)
top_agences

nom_agence
Bordeaux-Meriadeck      414
Marseille-Vieux-Port    323
Lyon-Part-Dieu          299
Paris-Centre            228
Lille-Grand-Place       214
Nantes-Commerce         200
Toulouse-Capitole       191
Nice-Massena            131
Name: id_transactions, dtype: int64

In [40]:
df.to_csv(r"C:\Users\nouha\Desktop\Dashboard_Gestion_des_Risques_pour_FinanceCore_SA\data\data.csv",index=False)